In [1]:
import torch
import h5py
from torch import nn, einsum
import torch.nn.functional as F
import math
from einops import rearrange
from einops.layers.torch import Rearrange

In [2]:
# Helper Functions
def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d

def calc_same_padding(kernel_size):
    pad = kernel_size // 2
    return (pad, pad - (kernel_size + 1) % 2)

In [3]:
# Helper Classes
class Swish(nn.Module):
    def forward(self, x):
        return x * x.sigmoid()

class GLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x):
        out, gate = x.chunk(2, dim=self.dim)
        return out * gate.sigmoid()

class DepthWiseConv1d(nn.Module):
    def __init__(self, channels, kernel_size, padding):
        super().__init__()
        self.padding = padding
        self.conv = nn.Conv1d(
            channels,
            channels,
            kernel_size,
            groups=channels,
            bias=False
        )

    def forward(self, x):
        x = F.pad(x, self.padding)
        return self.conv(x)

class Scale(nn.Module):
    def __init__(self, scale, fn):
        super().__init__()
        self.fn = fn
        self.scale = scale

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) * self.scale

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)

    def forward(self, x, **kwargs):
        x = self.norm(x)
        return self.fn(x, **kwargs)

In [4]:
# Relative Position Bias
class RelativePositionBias(nn.Module):
    def __init__(self, heads, max_distance=128):
        super().__init__()
        self.heads = heads
        self.max_distance = max_distance
        
        # 2*max_distance - 1 to cover all relative positions from -max_distance to +max_distance
        num_buckets = 2 * max_distance - 1
        self.relative_attention_bias = nn.Parameter(torch.zeros(heads, num_buckets))
        nn.init.normal_(self.relative_attention_bias, std=0.02)
    
    def forward(self, seq_len, device):
        # relative position matrix
        positions = torch.arange(seq_len, device=device)
        relative_positions = positions[None, :] - positions[:, None]  # [seq_len, seq_len]
        
        relative_positions = torch.clamp(
            relative_positions, 
            -self.max_distance + 1, 
            self.max_distance - 1
        )
        
        # non-negative indices
        relative_positions = relative_positions + self.max_distance - 1  # [seq_len, seq_len]
        
        # bias values
        bias = self.relative_attention_bias[:, relative_positions]  # [heads, seq_len, seq_len]
        
        return bias

In [5]:
# Feed Forward
class FeedForward(nn.Module):
    def __init__(
        self,
        dim,
        mult=4,
        dropout=0.
    ):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            Swish(),
            nn.Dropout(dropout),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [6]:
# Conformer Convolution Module
class ConformerConvModule(nn.Module):
    def __init__(self, dim, causal=False, expansion_factor=2, kernel_size=31, dropout=0.):
        super().__init__()

        inner_dim = dim * expansion_factor
        padding = calc_same_padding(kernel_size) if not causal else (kernel_size - 1, 0)

        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            Rearrange('b n c -> b c n'),
            nn.Conv1d(dim, inner_dim * 2, 1),
            GLU(dim=1),
            DepthWiseConv1d(inner_dim, kernel_size=kernel_size, padding=padding),
            nn.BatchNorm1d(inner_dim) if not causal else nn.Identity(),
            Swish(),
            nn.Conv1d(inner_dim, dim, 1),
            Rearrange('b c n -> b n c'),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [7]:
# Conformer Block (Without Attention)
class ConformerBlockNoAttn(nn.Module):
    def __init__(
        self,
        dim,
        ff_mult=4,
        conv_expansion_factor=2,
        conv_kernel_size=31,
        ff_dropout=0.,
        conv_dropout=0.,
        conv_causal=False
    ):
        super().__init__()

        self.ff1 = FeedForward(dim=dim, mult=ff_mult, dropout=ff_dropout)

        self.conv = ConformerConvModule(
            dim=dim,
            causal=conv_causal,
            expansion_factor=conv_expansion_factor,
            kernel_size=conv_kernel_size,
            dropout=conv_dropout
        )

        self.ff2 = FeedForward(dim=dim, mult=ff_mult, dropout=ff_dropout)

        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        """
        x: [B, T, C]
        """
        x = x + 0.5 * self.ff1(x)
        x = x + self.conv(x)
        x = x + 0.5 * self.ff2(x)
        return self.norm(x)

In [8]:
# Conformer (Without Attention)
class ConformerNoAttn(nn.Module):
    def __init__(
        self,
        dim,
        depth,
        ff_mult=4,
        conv_expansion_factor=2,
        conv_kernel_size=31,
        ff_dropout=0.,
        conv_dropout=0.,
        conv_causal=False
    ):
        super().__init__()
        self.dim = dim
        self.layers = nn.ModuleList([])

        for _ in range(depth):
            self.layers.append(ConformerBlockNoAttn(
                dim=dim,
                ff_mult=ff_mult,
                conv_expansion_factor=conv_expansion_factor,
                conv_kernel_size=conv_kernel_size,
                ff_dropout=ff_dropout,
                conv_dropout=conv_dropout,
                conv_causal=conv_causal
            ))

    def forward(self, x):
        for block in self.layers:
            x = block(x)
        return x

In [9]:
# ViT Attention with Learnable Relative Position Bias
class ViTAttention(nn.Module):
    def __init__(
        self,
        dim,
        heads=8,
        dim_head=64,
        dropout=0.0,
        max_distance=128
    ):
        super().__init__()
        inner_dim = heads * dim_head
        self.heads = heads
        self.scale = dim_head ** -0.5

        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        )

        # Replace RoPE with learnable relative position bias
        self.relative_position_bias = RelativePositionBias(heads, max_distance)

    def forward(self, x):
        b, n, _, h = *x.shape, self.heads
        
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(
            lambda t: rearrange(t, "b n (h d) -> b h n d", h=h),
            qkv
        )

        dots = einsum("b h i d, b h j d -> b h i j", q, k) * self.scale
        
        # Add learnable relative position bias
        relative_bias = self.relative_position_bias(n, x.device)  # [heads, seq_len, seq_len]
        dots = dots + relative_bias.unsqueeze(0)  # [b, heads, seq_len, seq_len]
        
        attn = dots.softmax(dim=-1)

        out = einsum("b h i j, b h j d -> b h i d", attn, v)
        out = rearrange(out, "b h n d -> b n (h d)")
        return self.to_out(out)

In [10]:
# Vision Transformer Block
class ViTBlock(nn.Module):
    def __init__(
        self,
        dim,
        heads=8,
        dim_head=64,
        ff_mult=4,
        dropout=0.
    ):
        super().__init__()
        
        self.attn = PreNorm(dim, ViTAttention(
            dim=dim,
            heads=heads,
            dim_head=dim_head,
            dropout=dropout
        ))
        
        self.ff = PreNorm(dim, FeedForward(
            dim=dim,
            mult=ff_mult,
            dropout=dropout
        ))

    def forward(self, x):
        x = x + self.attn(x)
        x = x + self.ff(x)
        return x

In [11]:
# Vision Transformer
class VisionTransformer(nn.Module):
    def __init__(
        self,
        dim,
        depth,
        heads=8,
        dim_head=64,
        ff_mult=4,
        dropout=0.
    ):
        super().__init__()
        self.layers = nn.ModuleList([])
        
        for _ in range(depth):
            self.layers.append(ViTBlock(
                dim=dim,
                heads=heads,
                dim_head=dim_head,
                ff_mult=ff_mult,
                dropout=dropout
            ))
        
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        for block in self.layers:
            x = block(x)
        return self.norm(x)

In [12]:
# Combined Model: Conformer (No Attn) + ViT
class ConformerViTHybrid(nn.Module):
    def __init__(
        self,
        dim,
        conformer_depth,
        vit_depth,
        conformer_ff_mult=4,
        conformer_conv_expansion=2,
        conformer_conv_kernel=31,
        vit_heads=8,
        vit_dim_head=64,
        vit_ff_mult=4,
        dropout=0.,
        conv_causal=False
    ):
        super().__init__()
        
        self.conformer = ConformerNoAttn(
            dim=dim,
            depth=conformer_depth,
            ff_mult=conformer_ff_mult,
            conv_expansion_factor=conformer_conv_expansion,
            conv_kernel_size=conformer_conv_kernel,
            ff_dropout=dropout,
            conv_dropout=dropout,
            conv_causal=conv_causal
        )
        
        self.vit = VisionTransformer(
            dim=dim,
            depth=vit_depth,
            heads=vit_heads,
            dim_head=vit_dim_head,
            ff_mult=vit_ff_mult,
            dropout=dropout
        )

    def forward(self, x):
        """
        x: [B, T, C]
        """
        x = self.conformer(x)
        x = self.vit(x)
        return x

In [13]:
with h5py.File("hdf5_data_final/t15.2023.08.11/data_train.hdf5", "r") as f:
    trial = f["trial_0000"]
    
    x = trial["input_features"][:]      # (T, 512)
    # y = trial["transcription"][:]        # (500,)
    class_ids = trial["seq_class_ids"][:]  # (500,)

print(x.shape)

(321, 512)


In [14]:
# Create model
model = ConformerViTHybrid(
    dim=512,
    conformer_depth=3,
    vit_depth=3,
    conformer_ff_mult=4,
    conformer_conv_expansion=2,
    conformer_conv_kernel=31,
    vit_heads=8,
    vit_dim_head=64,
    vit_ff_mult=4,
    dropout=0.1
)

In [15]:
x_f = [x]
x_f = torch.tensor(x_f)

# Forward pass
with torch.no_grad():
    y = model(x_f)
    
print("Input:", x_f.shape)
print("Output:", y.shape)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Input: torch.Size([1, 321, 512])
Output: torch.Size([1, 321, 512])
Model parameters: 26,891,752


/var/folders/ng/t1gkj_rd70zdnvqfk5m9rcxr0000gn/T/ipykernel_32116/1026834118.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  x_f = torch.tensor(x_f)


In [16]:
y = model(x_f)
loss = y.mean()
loss.backward()
bad = []
for name, p in model.named_parameters():
    if p.requires_grad and p.grad is None:
        bad.append(name)
print("No-gradient params:", bad)

No-gradient params: []


In [17]:
optimizer = torch.optim.Adafactor(model.parameters(), lr=1e-3)

target = x_f

for step in range(300):
    optimizer.zero_grad()
    y = model(x_f)
    loss = ((y - target) ** 2).mean()
    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        print(step, loss.item())

0 0.5306714773178101
25 0.29105839133262634
50 0.18795132637023926
75 0.13769792020320892
100 0.11092888563871384
125 0.09633887559175491
150 0.0871722623705864
175 0.08054863661527634
200 0.07546959817409515
225 0.07158561795949936
250 0.06821398437023163
275 0.06541834771633148
